In [2]:
from pathlib import Path
import json
import re
import unicodedata

In [3]:
EXCERPT_ID = "excerpt_000"

PROJECT_ROOT = Path("/workspace/storyfx")

SEGMENT_SPEC_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "composition_plans"
    / f"pred_{EXCERPT_ID}_segment_specification.json"
)

FA_PARSED_PATH = (
    PROJECT_ROOT
    / "data"
    / EXCERPT_ID
    / "alignment"
    / f"{EXCERPT_ID}_fa_parsed.txt"
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "schedules"
    / f"{EXCERPT_ID}_segment_schedule.json"
)


print("Segment specification:", SEGMENT_SPEC_PATH)
print("Forced alignment:", FA_PARSED_PATH)
print("Output:", OUTPUT_PATH)

Segment specification: \workspace\storyfx\outputs\composition_plans\pred_excerpt_000_segment_specification.json
Forced alignment: \workspace\storyfx\data\excerpt_000\alignment\excerpt_000_fa_parsed.txt
Output: \workspace\storyfx\outputs\schedules\excerpt_000_segment_schedule.json


In [4]:
def normalize_word(word):
    word = unicodedata.normalize("NFKC", word)

    # Normalize typographic apostrophes/quotes
    word = (
        word
        .replace("’", "'")
        .replace("‘", "'")
        .replace("“", '"')
        .replace("”", '"')
    )

    word = word.lower()

    # Keep letters/numbers/apostrophes.
    # Remove punctuation such as commas, periods, quotes, etc.
    word = re.sub(
        r"[^a-z0-9']+",
        "",
        word,
    )

    return word

In [5]:
def load_fa_words(path):
    text = Path(path).read_text(
        encoding="utf-8"
    ).strip()

    words = []

    for item in text.split():
        if "|" not in item:
            continue

        word, end_time = item.rsplit("|", 1)

        words.append(
            {
                "word": word,
                "normalized": normalize_word(word),
                "end_time": float(end_time),
            }
        )

    # Infer each word's start time from the preceding word's end.
    for i, word in enumerate(words):
        if i == 0:
            word["start_time"] = 0.0
        else:
            word["start_time"] = words[i - 1]["end_time"]

    return words

In [6]:
fa_words = load_fa_words(
    FA_PARSED_PATH
)

print("Aligned words:", len(fa_words))
print(fa_words[:10])

FileNotFoundError: [Errno 2] No such file or directory: '\\workspace\\storyfx\\data\\excerpt_000\\alignment\\excerpt_000_fa_parsed.txt'

In [7]:
def anchor_tokens(anchor):
    tokens = []

    for word in anchor.split():
        normalized = normalize_word(word)

        if normalized:
            tokens.append(normalized)

    return tokens

In [8]:
def find_anchor(
    anchor,
    fa_words,
    start_index=0,
):
    target = anchor_tokens(anchor)

    if not target:
        raise ValueError(
            f"Anchor contains no matchable words: {anchor}"
        )

    aligned = [
        word["normalized"]
        for word in fa_words
    ]

    target_length = len(target)

    for i in range(
        start_index,
        len(aligned) - target_length + 1,
    ):
        candidate = aligned[
            i:i + target_length
        ]

        if candidate == target:
            return i, i + target_length - 1

    raise ValueError(
        f"Could not match anchor:\n{anchor}"
    )

In [9]:
def quantify_segments(
    segment_spec_path,
    fa_parsed_path,
):
    segments = json.loads(
        Path(segment_spec_path).read_text(
            encoding="utf-8"
        )
    )

    fa_words = load_fa_words(
        fa_parsed_path
    )

    scheduled_segments = []

    search_index = 0

    for segment in segments:
        start_anchor = segment["start"]
        end_anchor = segment["end"]

        # --------------------------------------------
        # Match start anchor
        # --------------------------------------------

        start_first, start_last = find_anchor(
            start_anchor,
            fa_words,
            start_index=search_index,
        )

        # --------------------------------------------
        # Match end anchor
        # --------------------------------------------

        end_first, end_last = find_anchor(
            end_anchor,
            fa_words,
            start_index=start_first,
        )

        if end_last < start_first:
            raise ValueError(
                f"End anchor occurs before start anchor "
                f"for {segment['segment_id']}"
            )

        start_time = fa_words[
            start_first
        ]["start_time"]

        end_time = fa_words[
            end_last
        ]["end_time"]

        quantified = {
            **segment,
            "start_time": round(
                start_time,
                2,
            ),
            "end_time": round(
                end_time,
                2,
            ),
        }

        scheduled_segments.append(
            quantified
        )

        # Future segments must occur after this one.
        search_index = end_last + 1

    return scheduled_segments

In [ ]:
segment_schedule = quantify_segments(
    SEGMENT_SPEC_PATH,
    FA_PARSED_PATH,
)

segment_schedule

In [ ]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_PATH.write_text(
    json.dumps(
        segment_schedule,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    f"Segment schedule saved to: {OUTPUT_PATH}"
)